In [2]:
"""
BRFSS Summary Tables Extractor
================================
Extracts Total + demographic breakdowns from all BRFSS PHR summary sheets
into a clean 3-sheet Excel workbook.

Usage:
    pip install openpyxl pandas
    python brfss_extract.py

Inputs / Outputs (edit the two paths below):
    SRC  – path to your source .xlsx file
    OUT  – path for the output .xlsx file
"""

import re
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── CONFIGURE THESE TWO PATHS ─────────────────────────────────────────────────
SRC = "data/brfss/2024_PHR1_BRFSS_Summary_Tables.xlsx"   # <- your source file
OUT = "BRFSS_1Extracted.csv"                   # <- output file
# ─────────────────────────────────────────────────────────────────────────────

# Sheets to include in the "Disease Focus" tab.
# Edit this list to match your 11 (or any number of) target sheets.
DISEASE_SHEETS = [
    "Heart Disease", "Heart Attack", "Coronary Heart Disease", "CVD", "Stroke",
    "Diabetes", "COPD", "Arthritis", "Depressive Disorder", "Kidney Disease",
    "Current Asthma", "Skin Cancer", "Other Cancer", "Any Cancer",
    "Ever Told High BP", "Fair or Poor Health", "General Health",
    "Obese", "Overweight or Obese", "BMI 3 Categories",
    "Leisure Time PA", "Current Smoker", "Heavy Drinking", "Binge Drinking",
]

# For sheets where the FIRST response column is the "healthy/good" group,
# set the dict value to the 0-based index of the column you actually want.
# Format: "Sheet Name": column_index_of_percent  (3 = first %, 5 = second %, 7 = third %)
PRIMARY_COL_OVERRIDES = {
    # Example: these sheets list "None to <5 days" first; override to the 5+ column
    "Poor Physical Health 5+ Days":  5,
    "Poor Mental Health 5+ Days":    5,
    "Hlth Affected Activ 5+ Days":   5,
    "Poor Physical Health 14+ Day":  5,
    "Poor Mental Health 14+ Days":   5,
    "Hlth Affected Activ 14+ Days":  5,
}

# Sheets to skip entirely
SKIP_SHEETS = {"Index"}


# ── Styles ────────────────────────────────────────────────────────────────────
H1         = Font(name="Arial", bold=True, size=11, color="FFFFFF")
DT         = Font(name="Arial", size=10)
BD         = Font(name="Arial", bold=True, size=10)
FILL_BLUE  = PatternFill("solid", start_color="1F4E79")  # dark blue  – header
FILL_TOTAL = PatternFill("solid", start_color="D6E4F0")  # light blue – total rows
THIN       = Side(style="thin", color="BFBFBF")
BORDER     = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def _sc(cell, font=None, fill=None, align="left", bold=False):
    if font: cell.font  = font
    if fill: cell.fill  = fill
    cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=False)
    cell.border    = BORDER

def _hdr(ws, columns, widths):
    """Write a styled header row and set column widths."""
    for c, h in enumerate(columns, 1):
        cell = ws.cell(row=1, column=c, value=h)
        _sc(cell, font=H1, fill=FILL_BLUE, align="center")
    ws.row_dimensions[1].height = 28
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.auto_filter.ref = f"A1:{get_column_letter(len(columns))}1"
    ws.freeze_panes    = ws.cell(row=2, column=1)


NOTE_PAT = re.compile(r"^(Note:|Software:|Prepared)", re.IGNORECASE)

def parse_sheet(ws, sheet_name):
    """
    Parse one BRFSS summary sheet and return a dict with:
        topic, age_grp, question, categories, primary_cat, primary_col, data
    Returns None if the sheet is too short to be a data sheet.
    """
    rows = list(ws.iter_rows(values_only=True))
    if len(rows) < 10:
        return None

    topic    = str(rows[2][0]).strip() if rows[2][0] else ""
    age_grp  = str(rows[3][0]).strip() if rows[3][0] else ""
    question = str(rows[4][0]).strip() if rows[4][0] else ""

    # Row 7 (0-indexed) = response category names
    # Row 8            = column sub-headers (Demographic Groups / Sample Size / % / 95% CI …)
    cat_row = rows[7]

    # Build list of (col_index, category_label) for every response category
    categories = []
    for i in range(3, len(cat_row), 2):
        v = cat_row[i]
        if v and str(v).strip() not in ("", "None"):
            categories.append((i, str(v).strip()))

    # Default: first category's % column
    default_col = categories[0][0] if categories else 3
    primary_col = PRIMARY_COL_OVERRIDES.get(sheet_name, default_col)

    # Find the matching category label for the chosen column
    primary_cat = next(
        (label for idx, label in categories if idx == primary_col),
        categories[0][1] if categories else ""
    )

    # Data rows start at index 9
    data_rows      = []
    current_group  = ""
    for r in rows[9:]:
        # Stop at blank separator or footnote rows
        if r[0] is None and r[1] is None:
            break
        if NOTE_PAT.match(str(r[0] or "")):
            break

        grp    = str(r[0]).strip() if r[0] else ""
        subgrp = str(r[1]).strip() if r[1] else ""
        if grp:
            current_group = grp

        n   = r[2]
        pct = r[primary_col]     if len(r) > primary_col     else None
        ci  = r[primary_col + 1] if len(r) > primary_col + 1 else None

        data_rows.append({
            "group":   current_group,
            "subgroup": subgrp,
            "n":        n,
            "pct":      pct,
            "ci":       ci,
        })

    return {
        "topic":       topic,
        "age_grp":     age_grp,
        "question":    question,
        "categories":  categories,
        "primary_cat": primary_cat,
        "primary_col": primary_col,
        "data":        data_rows,
    }


def build_output(parsed_all):
    wb = Workbook()

    # ── Sheet 1: Summary (one Total row per topic) ────────────────────────────
    ws1 = wb.active
    ws1.title = "Summary - Totals"
    _hdr(ws1,
         ["Sheet Name", "Topic / Category", "Age Group", "Question / Variable",
          "Primary Outcome", "Total N", "Total %", "95% CI"],
         [28, 48, 20, 60, 28, 10, 12, 20])

    for sheet_name, parsed in parsed_all.items():
        total = next((d for d in parsed["data"] if d["subgroup"] == "Total"), None)
        row   = [
            sheet_name,
            parsed["topic"],
            parsed["age_grp"],
            parsed["question"],
            parsed["primary_cat"],
            total["n"]   if total else "",
            total["pct"] if total else "",
            total["ci"]  if total else "",
        ]
        r = ws1.max_row + 1
        for c, val in enumerate(row, 1):
            cell = ws1.cell(row=r, column=c, value=val)
            _sc(cell, font=DT, fill=FILL_TOTAL)

    # ── Sheet 2: Full Detail ──────────────────────────────────────────────────
    ws2 = wb.create_sheet("Full Detail - All Demographics")
    _hdr(ws2,
         ["Sheet Name", "Topic / Category", "Primary Outcome",
          "Demographic Group", "Subgroup", "Sample N", "% (Primary)", "95% CI"],
         [28, 48, 28, 22, 30, 10, 14, 20])

    for sheet_name, parsed in parsed_all.items():
        for d in parsed["data"]:
            is_total = (d["subgroup"] == "Total")
            r        = ws2.max_row + 1
            row      = [
                sheet_name, parsed["topic"], parsed["primary_cat"],
                d["group"], d["subgroup"], d["n"], d["pct"], d["ci"],
            ]
            for c, val in enumerate(row, 1):
                cell = ws2.cell(row=r, column=c, value=val)
                _sc(cell,
                    font=BD if is_total else DT,
                    fill=FILL_TOTAL if is_total else None)

    # ── Sheet 3: Disease Focus ────────────────────────────────────────────────
    ws3 = wb.create_sheet("Disease Focus - Total + Demog")
    _hdr(ws3,
         ["Topic / Category", "Primary Outcome",
          "Demographic Group", "Subgroup", "Sample N", "% (Primary)", "95% CI"],
         [50, 28, 22, 30, 10, 14, 20])

    prev_topic = None
    for sheet_name in DISEASE_SHEETS:
        if sheet_name not in parsed_all:
            continue
        parsed = parsed_all[sheet_name]
        for d in parsed["data"]:
            is_total = (d["subgroup"] == "Total")
            is_new   = (parsed["topic"] != prev_topic)
            r        = ws3.max_row + 1
            row      = [
                parsed["topic"] if is_new else "",
                parsed["primary_cat"],
                d["group"], d["subgroup"],
                d["n"], d["pct"], d["ci"],
            ]
            for c, val in enumerate(row, 1):
                cell = ws3.cell(row=r, column=c, value=val)
                _sc(cell,
                    font=BD if is_total else DT,
                    fill=FILL_TOTAL if is_total else None)
            if is_new:
                prev_topic = parsed["topic"]
        ws3.append([""] * 7)   # blank row between topics

    return wb


def main():
    print(f"Reading: {SRC}")
    wb_in      = load_workbook(SRC, read_only=True)
    parsed_all = {}

    for sheet_name in wb_in.sheetnames:
        if sheet_name in SKIP_SHEETS:
            continue
        parsed = parse_sheet(wb_in[sheet_name], sheet_name)
        if parsed:
            parsed_all[sheet_name] = parsed

    print(f"  Parsed {len(parsed_all)} sheets")

    wb_out = build_output(parsed_all)
    wb_out.save(OUT)
    print(f"Saved:  {OUT}")
    print("Output sheets:", [s.title for s in wb_out.worksheets])


if __name__ == "__main__":
    main()

Reading: data/brfss/2024_PHR1_BRFSS_Summary_Tables.xlsx
  Parsed 161 sheets
Saved:  BRFSS_1Extracted.csv
Output sheets: ['Summary - Totals', 'Full Detail - All Demographics', 'Disease Focus - Total + Demog']
